# 문항 1 : 네이버 VIBE Top 100 수집

## 네이버 VIBE 차트에서 오늘의 Top 100 순위를 수집하시오.

대상: https://vibe.naver.com/chart

추출 필드: 순위 / 곡명 / 아티스트

아티스트는 리스트로 담을 것 (협업곡은 여러 명)

Selenium 사용 금지

100건이 모두 수집되었는지 건수로 확인할 것

결과를 vibe_top100.csv로 저장할 것

### 결과 예시

[{'순위': 1, '곡명': 'All I Want for Christmas Is You', '아티스트': ['Mariah Carey']},
 {'순위': 2, '곡명': 'APT.', '아티스트': ['로제 (ROSÉ)', 'Bruno Mars']},
 ...]


In [1]:
import csv
import json
import re
import requests
from bs4 import BeautifulSoup

In [10]:
URL = "https://apis.naver.com/vibeWeb/musicapiweb/vibe/v1/chart/track/total?"

PARAMS = {
    "start": "1",
    "display": "100",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Referer" : "https://vibe.naver.com/chart/total",
    "Accept": "application/json"
}

res = requests.get(URL, headers=HEADERS, params=PARAMS)
res.status_code

200

In [15]:
if res.status_code == 200:
    data = res.json()
    track_list = data.get("response", {}).get("result", {}).get("chart", {}).get("items", {}).get("tracks", [])

print(f"가져온 노래 개수: {len(track_list)}개")

가져온 노래 개수: 100개


In [23]:
parsed_list = []
   
for index, track in enumerate(track_list, start=1):
    title = track.get('trackTitle')

    artists = []
    if 'artists' in track:
        artists = [artist.get("artistName") for artist in track["artists"]]
    else:
        artists = [track.get("artistName")]

    track_info = {
        '순위': index,
        '곡명': title,
        '아티스트': artists,
        }
        
    parsed_list.append(track_info)

print(len(parsed_list))


100


In [24]:
if parsed_list:
    filename = "vibe_top100.csv"
    with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
        fieldnames = ['순위', '곡명', '아티스트']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
            
        writer.writeheader()        
        writer.writerows(parsed_list)   
            
    print(f"성공적으로 '{filename}' 저장")

성공적으로 'vibe_top100.csv' 저장


# 문항 2 : 삼성전자 일별 시세 1년치 수집

## 네이버 금융에서 삼성전자(005930)의 일별 시세를 1년치 수집하시오.

대상: https://finance.naver.com/item/sise.naver?code=005930

추출 필드: 날짜 / 종가 / 전일비 / 시가 / 고가 / 저가 / 거래량

1년치 전량을 페이지네이션으로 수집할 것

숫자는 정제하여 숫자로 변환할 것

마지막 페이지를 자동으로 감지해 종료할 것

결과를 samsung_1y.csv로 저장할 것

### 결과 예시

[{'날짜': '2024.12.24', '종가': 54400, '전일비': '상승 900',
  '시가': 53700, '고가': 54500, '저가': 53600, '거래량': 11559385}, ...]

In [ ]:
URL = "https://finance.naver.com/item/sise_day.naver?"

def fetch(page):
    res = requests.get(URL, headers={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Whale/4.39.410.14 Safari/537.36",
    }, params={
        "code": "005930",
        "page": page,
    })
    
    if res.status_code == 200:
        return res.text
    else:
        print(f"{page}페이지 요청 실패 (상태코드: {res.status_code})")
    return None

def to_int(text):
    if not text:
        return 0
    clean_text = text.replace(",", "").strip()
    return int(clean_text) if clean_text.isdigit() else 0

def parse(data):
    html = data if isinstance(data, str) else data.get('innerHTML', '')
    soup = BeautifulSoup(html, "html.parser")

    parsed_list = []
    rows = soup.select('table.type2 tr[onMouseOver="mouseOver(this)"]')
    for row in rows:
        tds = row.select("td")

        if (len(tds)) <7:
            continue

        change_text = tds[2].get_text(strip=True)

        change_text = change_text.replace(",", "")
        if "상승" in change_text:
            change_text = change_text.replace("상승", "상승 ")
        elif "하락" in change_text:
            change_text = change_text.replace("하락", "하락 ")

        parsed_list.append({
            "날짜": tds[0].get_text(strip=True), 
            "종가": to_int(tds[1].get_text(strip=True)),
            "전일비": change_text,
            "시가": to_int(tds[3].get_text(strip=True)),
            "고가": to_int(tds[4].get_text(strip=True)),
            "저가": to_int(tds[5].get_text(strip=True)),
            "거래량": to_int(tds[6].get_text(strip=True)),
        })
    return parsed_list

In [31]:
import time

In [35]:
if __name__ == "__main__":
    all_data = []
    page = 1

    latest_date = None
    target_past_date = None

    while True:
        html_content = fetch(page)
        if not html_content:
            break

        page_data = parse(html_content)

        if not page_data:
            break

        all_data.extend(page_data)

        if len(all_data) >= 250:
            all_data = all_data[:250]
            break

        page += 1
        time.sleep(0.5)

    
    print(all_data[:2])
    print(len(all_data))


[{'날짜': '2026.08.21', '종가': 281500, '전일비': '상승 10500', '시가': 267000, '고가': 285000, '저가': 266000, '거래량': 27672192}, {'날짜': '2026.08.20', '종가': 271000, '전일비': '상승 23500', '시가': 257000, '고가': 273000, '저가': 252500, '거래량': 26095919}]
250


In [36]:
if all_data:
    filename = "samsung_1y.csv"
    with open(filename, 'w', encoding='utf-8-sig', newline='') as ff:
        fieldnames = ['날짜', '종가', '전일비', '시가', '고가', '저가', '거래량']
        writer = csv.DictWriter(ff, fieldnames=fieldnames)
            
        writer.writeheader()        
        writer.writerows(all_data)   
            
    print(f"성공적으로 '{filename}' 저장")

성공적으로 'samsung_1y.csv' 저장


# 문항 3 : 네이버 뉴스 검색기 함수 만들기 (스크롤 포함) - 난이도 상

## 키워드를 받아 네이버 뉴스 검색 결과를 수집하는 함수를 작성하시오.

대상: https://search.naver.com/search.naver?ssc=tab.news.all

crawl_naver_news(keyword, page) — 아래로 스크롤해야 나오는 결과까지 수집, ※page 수 만큼 스크롤

추출 필드: 제목 / 언론사 / 링크 / 요약

Selenium 사용 금지. 스크롤이 유발하는 요청을 Network에서 찾아 재현할 것

결과를 news_{keyword}.csv로 저장할 것

### 결과 예시

[{'제목': '뉴욕증시, CPI 예상 부합에 상승 출발…나스닥 0.9%↑새 창 열림',
  '언론사': '조선비즈',
  '링크': 'https://biz.chosun.com/international/international_general/2026/08/12/N3YG7LVYFJH2FHXQP2RRFUFNS4/?utm_source=naver&utm_medium=original&utm_campaign=biz',
  '요약': '미국의 인플레이션 지표가 시장 예상에 부합하고 인공지능(AI) 관련 기업들의 실적 호조가 이어지면서 뉴욕증시가 상승 출발했다. 12일(현지시각) 뉴욕증권거래소(NYSE)에 따르면 이날 다우존스30산업평균지수는 전장보다 5.6포인트(0.01%) 오른 5만 3797.47에 거래를 시작했다. 스탠더드앤드푸어스(S&P)... '},
 {'제목': '‘하나의 영혼’이 아리스토텔레스 우정 명언? AI시대, 원전을 펼쳐라[...새 창 열림',
  '언론사': '동아일보',
  '링크': 'https://www.donga.com/news/Opinion/article/all/20260812/134468253/2',
  '요약': '《 AI가 재생산하는 인간의 오류 인공지능(AI) 시대에는 검색 기능을 잘 활용하면 자료를 손쉽게 찾을 수 있다. 그러나 자칫 AI가 내놓은 답에 지나치게 의존하다 보면 낭패를 볼 수도 있다. 우정을 다룬 가장 유명한 철학서는 아리스토텔레스의 ‘니코마코스 윤리학’이다. 친애, 우애, 우정으로 번역되는... '}, ...]

In [ ]:
from urllib.parse import unquote

In [61]:
URL = "https://search.naver.com/search.naver?ssc=tab.news.all"

payload_str = 'abt=null&cluster_rank=52&de=&ds=&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22shopping_top%22%2C%22source%22%3A%22TOS%22%7D%2C%22sub%22%3A%5B%7B%22name%22%3A%22shopping%22%7D%5D%7D%7D&nso=so%3Ar%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=%EC%95%84%EA%B8%B0%EC%98%B7&query_original=&rev=0&service_area=&sm=tab_smr&sort=0&spq=0&ssc=tab.news.all&start=1'
payload = [p.split('=') for p in payload_str.split('&')]
payload = {k: unquote(v) for k, v in payload}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Referer": "https://search.naver.com/",
}

res = requests.get(URL, headers=HEADERS, params=payload)

res.status_code

200

In [85]:
all_news = []

def crawl_naver_news(keyword, page):
    
    for i in range(page):
        start_num = 1 + (i * 10)

        payload_str = 'abt=null&cluster_rank=52&de=&ds=&eid=&field=0&force_original=&is_dts=0&is_sug_officeid=0&mynews=0&news_office_checked=&nlu_query=&nqx_theme=%7B%22theme%22%3A%7B%22main%22%3A%7B%22name%22%3A%22shopping_top%22%2C%22source%22%3A%22TOS%22%7D%2C%22sub%22%3A%5B%7B%22name%22%3A%22shopping%22%7D%5D%7D%7D&nso=so%3Ar%2Cp%3Aall%2Ca%3Aall&nx_and_query=&nx_search_hlquery=&nx_search_query=&nx_sub_query=&office_category=&office_section_code=0&office_type=0&pd=0&photo=0&qdt=0&query=%EC%95%84%EA%B8%B0%EC%98%B7&query_original=&rev=0&service_area=&sm=tab_smr&sort=0&spq=0&ssc=tab.news.all&start=1'
        payload = [p.split('=') for p in payload_str.split('&')]
        params = {k: unquote(v) for k, v in payload} 

        params["start"] = str(start_num)
        params["query"] = keyword

        try:
            res = requests.get(URL, headers=HEADERS, params=params)
            if res.status_code != 200:
                continue
        except Exception as e:
            print(f"네트워크 연결 실패: {e}")
            break

        try:
            json_str = re.search(r"\{.*\}", res.text, re.DOTALL).group(0)
            html_content = json.loads(json_str).get("contents", "")
        except Exception:
            html_content = res.text

        soup = BeautifulSoup(html_content, "html.parser")
        news_items = soup.select('div[class*="sds-comps-base-layout sds-comps-full-layout"]')

        if not news_items:
            break

        for item in news_items:
            link_tag = item.select_one('a[data-heatmap-target=".tit"]')
            if not link_tag:
                link_tag = item.select_one('a[data-heatmap-target=".body"]')
    
            link = link_tag.get("href", "") if link_tag else ""


            title_tag = item.select_one('[class*="sds-comps-text-type-headline1"]')
            title = ""
            if title_tag:
                title = title_tag.get_text(strip=True).replace("새 창 열림", "").strip()

            press_tag = soup.select_one('[class*="sds-comps-text-type-body2"]')
            press = press_tag.get_text(strip=True).replace("새 창 열림", "").strip() if press_tag else "알 수 없음"

            dsc_tag = item.select_one('[class*="sds-comps-text-type-body1"]')
            summary = ""
            if dsc_tag:
                summary = dsc_tag.get_text(strip=True).replace("새 창 열림", "").strip()

            
            if title and link:
                all_news.append({
                    "제목": title, 
                    "언론사": press, 
                    "링크": link, 
                    "요약": summary
                    })

        time.sleep(0.5) 

    if all_news:
        filename = f"news_{keyword}.csv"
        with open(filename, 'w', encoding='utf-8-sig', newline='') as fff:
            fieldnames = ['제목', '언론사', '링크', '요약']
            writer = csv.DictWriter(fff, fieldnames=fieldnames)
            
            writer.writeheader()        
            writer.writerows(all_news)   
            
        print(f"성공적으로 '{filename}' 저장")

    return all_news

In [86]:
if __name__ == "__main__":
    crawl_naver_news(keyword="오디세이", page=5)

성공적으로 'news_오디세이.csv' 저장
